# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane:** CTR / Engagement Opportunity Scoring  
**Author:** Chashman Aslam ·
**Warehouse snapshot:** FlyRank internship-warehouse v20260703  


## 0. Setup — authenticate and connect

In [1]:
import os, duckdb, pandas as pd
from IPython.display import display

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

assert HF_TOKEN, "Set HF_TOKEN in Colab Secrets (key panel) before running."

con = duckdb.connect()
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{HF_TOKEN}')")

PERF  = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
MONTH = '2026-03'

print("Connected. Working month:", MONTH)

Connected. Working month: 2026-03


---
## 1. Unit of analysis + time window

**One row =** one content item (page) for one client on one report date.  
The three columns that jointly identify a row: `(report_date, client_hash_id, content_hash_id)`.

**Table used:** `fact_content_daily_performance` — daily fact table partitioned by `month=YYYY-MM`.

**Time window:** Mid-panel month `2026-03` for all development. The final month `2026-06` is sealed and never touched.

**What I predict (label proxy):**  
Whether a page is a CTR opportunity — pages that already appear in Google Search (`gsc_impressions > 0`) but whose CTR (`gsc_clicks / gsc_impressions`) is low relative to their average position. The model ranks pages by how much CTR lift is plausible given their current position and impression volume.

**One thing deliberately excluded:**  
GA4 columns (`ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`) are excluded. Many rows have `ga4_data_available = False` (GSC-only clients), making GA4 features sparse and unreliable for a lane that must score all pages uniformly.

In [2]:
# Peek at columns and 3 sample rows
sample = con.sql(f"""
    SELECT *
    FROM read_parquet('{PERF}', hive_partitioning=true)
    WHERE month = '{MONTH}'
    LIMIT 3
""").df()

print("Columns:", list(sample.columns))
display(sample)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


---
## 2. Fields: feature / label / context / excluded

| Bucket | Field(s) | Role |
|--------|----------|------|
| **Label proxy** | `gsc_clicks`, `gsc_impressions` | CTR = gsc_clicks / gsc_impressions is the outcome. Used raw to construct label; not fed directly as a feature. |
| **Features** | `gsc_avg_position`, `gsc_impressions`, `gsc_clicks`, `gsc_data_available`, `scroll_events` | Position drives CTR by Google ranking mechanics. Impressions = reach. Clicks = momentum. Availability flag controls which rows are usable. Scroll events = engagement proxy from GSC-available clients. |
| **Context** | `report_date`, `client_hash_id`, `content_hash_id`, `month` | Identifiers and partition keys — used for grouping and joins, not fed to model. |
| **Excluded** | `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec` | GA4 columns are excluded — large fraction of clients have `ga4_data_available = False`, making these features unreliable across the full slice. |

**Why CTR as a raw column is not a feature:**  
CTR = gsc_clicks ÷ gsc_impressions. Using the same-period CTR as a feature lets the model trivially reconstruct the label — that is leakage. I use `gsc_avg_position` and `gsc_impressions` instead, which are knowable before observing click outcome.

In [3]:
# Field availability check
schema_check = con.sql(f"""
    SELECT
        COUNT(*)                        AS total_rows,
        COUNT(gsc_clicks)               AS gsc_clicks_non_null,
        COUNT(gsc_impressions)          AS gsc_impr_non_null,
        COUNT(gsc_avg_position)         AS gsc_pos_non_null,
        COUNT(scroll_events)            AS scroll_non_null,
        ROUND(AVG(gsc_avg_position),2)  AS mean_position,
        ROUND(AVG(gsc_clicks * 1.0 /
            NULLIF(gsc_impressions,0)),4) AS mean_ctr
    FROM read_parquet('{PERF}', hive_partitioning=true)
    WHERE month = '{MONTH}'
""").df()

display(schema_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_clicks_non_null,gsc_impr_non_null,gsc_pos_non_null,scroll_non_null,mean_position,mean_ctr
0,9841378,9841378,9841378,3611061,6822637,15.83,0.0031


---
## 3. Verify with three queries

### Query 1 — Grain verification
One row really is one `(report_date, client_hash_id, content_hash_id)` triple.

In [4]:
# QUERY 1: Grain check — max_count_per_triple must equal 1
grain_check = con.sql(f"""
    SELECT
        COUNT(*)                                                        AS total_rows,
        COUNT(DISTINCT report_date || client_hash_id || content_hash_id) AS distinct_triples,
        MAX(cnt)                                                        AS max_count_per_triple
    FROM (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS cnt
        FROM read_parquet('{PERF}', hive_partitioning=true)
        WHERE month = '{MONTH}'
        GROUP BY report_date, client_hash_id, content_hash_id
    ) sub
""").df()

print("QUERY 1 — Grain verification (max_count_per_triple must = 1):")
display(grain_check)
assert grain_check['max_count_per_triple'][0] == 1, "GRAIN VIOLATION!"
print("✓ Grain confirmed: one row = one (report_date, client_hash_id, content_hash_id)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

QUERY 1 — Grain verification (max_count_per_triple must = 1):


,total_rows,distinct_triples,max_count_per_triple
0,9841378,9841378,1


✓ Grain confirmed: one row = one (report_date, client_hash_id, content_hash_id)


### Query 2 — Row count and date span

In [5]:
# QUERY 2: Row count + date span for working month
count_span = con.sql(f"""
    SELECT
        COUNT(*)                        AS row_count,
        COUNT(DISTINCT client_hash_id)  AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_pages,
        MIN(report_date)                AS earliest_date,
        MAX(report_date)                AS latest_date,
        COUNT(DISTINCT report_date)     AS n_dates
    FROM read_parquet('{PERF}', hive_partitioning=true)
    WHERE month = '{MONTH}'
""").df()

print(f"QUERY 2 — Slice summary for month={MONTH}:")
display(count_span)

QUERY 2 — Slice summary for month=2026-03:


,row_count,n_clients,n_pages,earliest_date,latest_date,n_dates
0,9841378,55,331437,2026-03-01,2026-03-31,31


### Query 3 — Availability: `gsc_data_available IS TRUE`

In [6]:
# QUERY 3: Availability — how many rows survive IS TRUE filter
availability = con.sql(f"""
    SELECT
        COUNT(*)                                        AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)
                                                        AS gsc_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                           AND gsc_impressions > 0)     AS gsc_with_impressions,
        ROUND(100.0 *
            COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)
            / COUNT(*), 2)                              AS availability_pct
    FROM read_parquet('{PERF}', hive_partitioning=true)
    WHERE month = '{MONTH}'
""").df()

print(f"QUERY 3 — Availability (gsc_data_available IS TRUE) for month={MONTH}:")
display(availability)
print("\nOnly rows where gsc_data_available IS TRUE are usable for CTR opportunity scoring.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

QUERY 3 — Availability (gsc_data_available IS TRUE) for month=2026-03:


,total_rows,gsc_available_rows,gsc_with_impressions,availability_pct
0,9841378,3611061,3611061,36.69



Only rows where gsc_data_available IS TRUE are usable for CTR opportunity scoring.


---
## 3b. Five-feature frame

Built from `month=2026-03`. Every feature gets an *"available when?"* justification.

In [7]:
# Five-feature frame — GSC-available rows only
feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,

        -- Feature 1: avg_position (GSC-reported)
        gsc_avg_position                                         AS avg_position,

        -- Feature 2: impressions volume
        gsc_impressions                                          AS impressions,

        -- Feature 3: clicks
        gsc_clicks                                               AS clicks,

        -- Feature 4: scroll_events (engagement signal)
        COALESCE(scroll_events, 0)                               AS scroll_events,

        -- Feature 5: impression rank within client-day
        RANK() OVER (
            PARTITION BY client_hash_id, report_date
            ORDER BY gsc_impressions DESC
        )                                                        AS impression_rank,

        -- Label (kept separate — NOT used as feature)
        CASE
            WHEN gsc_impressions > 0
             AND (gsc_clicks * 1.0 / gsc_impressions) < 0.03
             AND gsc_avg_position <= 20
            THEN 1 ELSE 0
        END                                                      AS label_ctr_opportunity

    FROM read_parquet('{PERF}', hive_partitioning=true)
    WHERE month = '{MONTH}'
      AND gsc_data_available IS TRUE
      AND gsc_impressions > 0
    LIMIT 10000
""").df()

print(f"Feature frame shape: {feature_frame.shape}")
display(feature_frame.head(10))
print("\nLabel distribution:")
print(feature_frame['label_ctr_opportunity'].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (10000, 9)


,client_hash_id,content_hash_id,report_date,avg_position,impressions,clicks,scroll_events,impression_rank,label_ctr_opportunity
0,client_0797ff3a1fc9a6a5,content_be06033d30b49299,2026-03-07,58.741379,116,0,0,1,0
1,client_0797ff3a1fc9a6a5,content_a7a5f0d74ec03ce8,2026-03-07,8.287234,94,3,0,2,0
2,client_0797ff3a1fc9a6a5,content_fa84e03e3d5e2fa2,2026-03-07,10.354839,62,0,0,3,1
3,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,2026-03-07,13.333333,39,0,0,4,1
4,client_0797ff3a1fc9a6a5,content_7beb639d1052e49e,2026-03-07,8.037037,27,0,0,5,1
5,client_0797ff3a1fc9a6a5,content_1207efddce873942,2026-03-07,13.708333,24,0,0,6,1
6,client_0797ff3a1fc9a6a5,content_a0a6b37ae2f9a09c,2026-03-07,9.142857,14,0,0,7,1
7,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,2026-03-07,10.833333,12,0,0,8,1
8,client_0797ff3a1fc9a6a5,content_a08f993faabe02ea,2026-03-07,9.400000,5,0,0,9,1
9,client_0797ff3a1fc9a6a5,content_04c67f3541177192,2026-03-07,10.400000,5,0,0,9,1



Label distribution:
label_ctr_opportunity
1    6083
0    3917
Name: count, dtype: int64


### Feature availability justification

| # | Feature | Available when? |
|---|---------|----------------|
| 1 | `avg_position` | Knowable at decision moment because Google Search Console reports average position for dates already elapsed — a past measurement, not a future outcome. |
| 2 | `impressions` | Knowable at decision moment because GSC impression counts are logged for dates already elapsed; no future signal crosses in. |
| 3 | `clicks` | Knowable at decision moment because GSC click counts are historical measurements from the same past window — they precede any action taken on the recommendation. |
| 4 | `scroll_events` | Knowable at decision moment because scroll events are logged by the analytics layer for past dates; zero-filled where unavailable rather than imputed forward. |
| 5 | `impression_rank` | Knowable at decision moment because it is derived entirely from historical impressions within the same past window — a relative rank computed from already-observed data. |

---
## 3c. The Trap — deliberate label leakage experiment

In [8]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

clean_cols = ['avg_position', 'impressions', 'clicks', 'scroll_events', 'impression_rank']
label_col  = 'label_ctr_opportunity'

df = feature_frame.dropna(subset=clean_cols + [label_col])
X_clean = df[clean_cols]
y = df[label_col]

clf = GradientBoostingClassifier(n_estimators=50, random_state=42)

# HONEST score
score_honest = cross_val_score(clf, X_clean, y, cv=3, scoring='roc_auc').mean()
print(f"Honest AUC (clean features only):   {score_honest:.4f}")

# LEAKED score — add CTR directly (it IS what defines the label)
df_leak = df.copy()
df_leak['LEAKED_ctr'] = df_leak['clicks'] / df_leak['impressions'].replace(0, np.nan)
df_leak['LEAKED_ctr'] = df_leak['LEAKED_ctr'].fillna(0)

X_leaked = df_leak[clean_cols + ['LEAKED_ctr']]
score_leaked = cross_val_score(clf, X_leaked, y, cv=3, scoring='roc_auc').mean()
print(f"Leaked AUC (ctr added as feature):  {score_leaked:.4f}  <- artificially inflated")
print(f"AUC inflation from leakage:         +{score_leaked - score_honest:.4f}")

# DELETE leaked column
del df_leak
print("\n✓ Leaked column deleted. Honest AUC is the number that matters.")
print(f"  Final honest AUC: {score_honest:.4f}")

Honest AUC (clean features only):   0.9881
Leaked AUC (ctr added as feature):  0.9884  <- artificially inflated
AUC inflation from leakage:         +0.0003

✓ Leaked column deleted. Honest AUC is the number that matters.
  Final honest AUC: 0.9881


**What happened:**  
`LEAKED_ctr` = `gsc_clicks / gsc_impressions` — mathematically identical to the quantity used to define `label_ctr_opportunity`. Adding it lets the model trivially reconstruct the label, pushing AUC toward 1.0. The column looks innocent (CTR is a natural SEO metric) but at decision time we would not yet have the same-period CTR that was used to assign the label. The honest AUC is lower but trustworthy.

---
## 4. Data limits

**Named limitation of this slice:**

> *The panel is unbalanced: clients have different history depths in the warehouse. A page with 2 months of history and a page with 18 months of history both appear in March 2026, but their `avg_position` and impression volume are not comparable without controlling for client tenure.*

Additional limits:
- **GA4 sparsity:** Many clients have `ga4_data_available = False` — engagement signals like session depth are absent for a large fraction of rows, forcing exclusion of GA4 features from a universal scorer.
- **Impression floor:** Google anonymizes very low-impression queries. Pages with thin traffic are partially unobservable in GSC.
- **Label is a proxy, not ground truth:** `label_ctr_opportunity` is a rule-based threshold on observed CTR vs position. It measures under-performance, not the true ceiling of CTR if content were improved.
- **No query-level signal:** `fact_content_query_90d` (keyword-level data) is excluded this week — richer signals deferred to modeling weeks.

In [9]:
# Illustrate GA4 sparsity — the main excluded feature group
sparsity = con.sql(f"""
    SELECT
        COUNT(*)                                                   AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)         AS ga4_available,
        COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE)     AS ga4_unavailable,
        ROUND(100.0 *
            COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)
            / COUNT(*), 1)                                         AS ga4_available_pct
    FROM read_parquet('{PERF}', hive_partitioning=true)
    WHERE month = '{MONTH}'
""").df()

print("GA4 availability — why GA4 columns are excluded:")
display(sparsity)

GA4 availability — why GA4 columns are excluded:


,total_rows,ga4_available,ga4_unavailable,ga4_available_pct
0,9841378,413966,9427412,4.2


---
## Self-check

- [x] Every section filled — markdown thinking AND the code that backs it
- [x] Notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Committed to repo under `work/notebooks/` — repo URL submitted on the card

---
**Contract summary (five plain-words answers):**

1. **One row =** one content page for one client on one report date.
2. **Table:** `fact_content_daily_performance`, working month `2026-03`.
3. **Time window:** March 2026 (mid-panel). Sealed test month June 2026 untouched.
4. **Predict / rank:** Pages with impressions but low CTR relative to their position — operationalized as `gsc_impressions > 0 AND CTR < 0.03 AND position <= 20`.
5. **Deliberately excluded:** GA4 columns (too sparse across clients) and same-period CTR as a direct feature (label-derived leakage).